In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: SETUP & LIBRARIES
# ─────────────────────────────────────────────────────────────
!pip install statsmodels
!pip install xgboost

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

print("✅ Cell 1 Complete: All libraries loaded.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2A: DATA LOADING & INITIAL FEATURES
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ================== CONFIGURATION ==================
# 1. Set your robot type: '3', '4', '5', '6', or 'panda'
ROBOT_TYPE = "3"

# 2. Automatically determine DOF and File Name
if ROBOT_TYPE == "panda":
    DOF = 7
    FILE_NAME = "panda.csv"
else:
    DOF = int(ROBOT_TYPE)
    FILE_NAME = f"{DOF}dof.csv"  # Removed the '_500' to match your actual files!

df = pd.read_csv(FILE_NAME)
print(f"Loaded {FILE_NAME}. Original rows: {len(df)}")

# ================== 1. TRAJECTORY FILTERING ==================
successful_episodes = df[df["is_success"] == 1]["episode"].unique()
df = df[df["episode"].isin(successful_episodes)].copy()
print(f"Filtered for successful episodes. Remaining rows: {len(df)}")

df = df.sort_values(["episode", "step_count"])

# ================== 2. SPATIAL FEATURES ==================
df["ee_error_x"] = df["ent_0_x"] - 0.5
df["ee_error_y"] = df["ent_0_y"] - 0.0
df["ee_error_z"] = df["ent_0_z"] - 0.5
df["ee_dist"] = np.sqrt(df["ee_error_x"]**2 + df["ee_error_y"]**2 + df["ee_error_z"]**2)

# ================== 3. TIME DELTA (dt) CALCULATION ==================
df["dt"] = df.groupby("episode")["step_time"].diff().fillna(0)

print("✅ Cell 2A Complete: Initial features and time deltas calculated.")

In [ ]:
# # ─────────────────────────────────────────────────────────────
# # CELL 2B: OUTLIER REMOVAL
# # ─────────────────────────────────────────────────────────────

# # We look for anomalies in end-effector distance or rewards which indicate physics glitches
# outlier_cols = ["ee_dist", "reward"]

# print(f"Rows before outlier removal: {len(df)}")

# # Apply IQR filtering
# for col in outlier_cols:
#     if col in df.columns:
#         Q1 = df[col].quantile(0.25)
#         Q3 = df[col].quantile(0.75)
#         IQR = Q3 - Q1

#         # Define bounds (1.5 is standard, 3.0 is for extreme outliers only)
#         lower_bound = Q1 - 2.5 * IQR
#         upper_bound = Q3 + 2.5 * IQR

#         # Filter the dataframe
#         df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]

# print(f"Rows after outlier removal: {len(df)}")
# print("✅ Cell 2B Complete: Outliers removed.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2C: ADVANCED KINEMATIC ENGINEERING & DATA PREP
# ─────────────────────────────────────────────────────────────

# Safe time delta to prevent division by zero
dt_safe = np.where(df["dt"] <= 0, 1e-5, df["dt"])

# ================== 1. BASE KINEMATIC FEATURES ==================
episode_len = df.groupby("episode").size()
df["episode_len"] = df["episode"].map(episode_len)
df["progress"] = df["step_count"] / df["episode_len"]

for i in range(DOF):
    # Trigonometric Encodings
    df[f"sin_joint_{i}"] = np.sin(df[f"joint_{i}"])
    df[f"cos_joint_{i}"] = np.cos(df[f"joint_{i}"])

    # True Velocity: dx/dt
    joint_diff = df[f"joint_{i}"] - df.groupby("episode")[f"joint_{i}"].shift(1).fillna(df[f"joint_{i}"])
    df[f"vel_{i}"] = joint_diff / dt_safe

    # True Acceleration: dv/dt
    vel_diff = df[f"vel_{i}"] - df.groupby("episode")[f"vel_{i}"].shift(1).fillna(0)
    df[f"accel_{i}"] = vel_diff / dt_safe

    # 🌟 NEW: Jerk (da/dt) -> Crucial for predicting sharp action changes
    accel_diff = df[f"accel_{i}"] - df.groupby("episode")[f"accel_{i}"].shift(1).fillna(0)
    df[f"jerk_{i}"] = accel_diff / dt_safe

    # 🌟 NEW: Kinetic Cross-Features (Velocity * Acceleration)
    df[f"vel_accel_{i}"] = df[f"vel_{i}"] * df[f"accel_{i}"]

    # 🌟 NEW: Deeper Memory Lags (What happened 2 steps ago?)
    df[f"prev_action_{i}"] = df.groupby("episode")[f"action_{i}"].shift(1).fillna(0)
    df[f"prev_action_2_{i}"] = df.groupby("episode")[f"action_{i}"].shift(2).fillna(0)
    df[f"prev_vel_{i}"] = df.groupby("episode")[f"vel_{i}"].shift(1).fillna(0)

    # Targets (Deltas)
    df[f"delta_action_{i}"] = df[f"action_{i}"] - df[f"prev_action_{i}"]

# ================== 2. DYNAMIC INPUTS & OUTPUTS ==================
OUTPUT_COLS = [f"delta_action_{i}" for i in range(DOF)]
raw_actions = [f"action_{i}" for i in range(DOF)]
raw_joints = [f"joint_{i}" for i in range(DOF)]

# We explicitly keep engineered features, but exclude raw controls & metadata
exclude_cols = OUTPUT_COLS + raw_actions + raw_joints + [
    'episode', 'step_count', 'reward', 'done', 'is_success',
    'episode_len', 'ent_0_x', 'ent_0_y', 'ent_0_z', 'step_time', 'dt'
]

INPUT_ALL = [col for col in df.columns if col not in exclude_cols]

X = df[INPUT_ALL].copy()
y = df[OUTPUT_COLS].copy()

print(f"✅ Cell 2C Complete. Pipeline configured for {DOF}-DOF robot.")
print(f"Total input features: {len(INPUT_ALL)}")
print(f"Total outputs being predicted: {len(OUTPUT_COLS)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: DATA SPLITTING & SCALING
# ─────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Cell 4 Complete: Data is split and standardized.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: TRAIN 6 MODELS (LR, SVM, RF, XGB, MLP, KNN)
# ─────────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

print("Training all models side-by-side...\n")

# 1. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
print("- Linear Regression trained.")

# 2. Support Vector Machine
svm_base = SVR(kernel='rbf', C=1.0, gamma='scale')
svm_model = MultiOutputRegressor(svm_base)
svm_model.fit(X_train_scaled, y_train)
print("- SVM trained.")

# 3. Random Forest
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model = MultiOutputRegressor(rf_base)
rf_model.fit(X_train_scaled, y_train)
print("- Random Forest trained.")

# 4. XGBoost
xgb_base = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6, random_state=42, n_jobs=-1)
xgb_model = MultiOutputRegressor(xgb_base)
xgb_model.fit(X_train_scaled, y_train)
print("- XGBoost trained.")

print("\n✅ Cell 5 Complete.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 6: THE FINAL REPORT CARD (ALL 4 MODELS)
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import r2_score

def evaluate_model(model_name, model):
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)

    print(f"--- {model_name} ---")
    print(f"  R-Squared Score : {r2:.4f}\n")

print(f"=== {DOF}DOF ===\n")
evaluate_model("Linear Regression", lr_model)
evaluate_model("Support Vector Machine", svm_model)
evaluate_model("Random Forest", rf_model)
evaluate_model("XGBoost", xgb_model)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 7: SAVE TRAINED MODELS (DIRECT TO GOOGLE DRIVE)
# ─────────────────────────────────────────────────────────────
import os
import pickle
from google.colab import drive

# 1. Mount your Google Drive (Follow the pop-up prompt to grant permission)
drive.mount('/content/drive')

# 2. Define the path pointing straight to your MyDrive folder
save_dir = f"/content/drive/MyDrive/Gazebo_Models_{DOF}DOF/"
os.makedirs(save_dir, exist_ok=True)

# Create a dictionary mapping names to your trained model objects
models_to_save = {
    "linear_regression": lr_model,
    "support_vector_machine": svm_model,
    "xgboost": xgb_model,
    "random_forest": rf_model,
    "scaler": scaler
}

print(f"Saving models for {DOF}-DOF to Google Drive directory: '{save_dir}'...\n")

for model_name, model_obj in models_to_save.items():
    # Safely check if the model object exists in memory
    if model_obj is not None:
        file_path = os.path.join(save_dir, f"{model_name}.pkl")
        with open(file_path, "wb") as f:
            pickle.dump(model_obj, f)
        print(f"✅ Saved to Drive: {model_name}.pkl")
    else:
        print(f"⚠️ Warning: Model '{model_name}' was not found/trained. Skipping.")

print("\n🎉 All available models and preprocess scalers successfully archived in your Google Drive!")